# PART 01 · 5장 판다스 집계와 계산

교재 121~150쪽 본문 소스 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

이번 장에서는 주문 데이터와 고객 데이터를 이용해 수치형 컬럼을 계산하고, 범주별로 묶어 집계하며, 두 데이터프레임을 병합하는 방법을 학습한다. 같은 데이터는 장 앞부분에서 한 번만 불러오고 뒤의 항목에서는 orders와 customers 데이터프레임을 이어서 사용한다. 실습 경로는 모두 data 폴더를 기준으로 작성한다.

In [1]:
import pandas as pd
import numpy as np
orders = pd.read_csv('data/part01_pandas/pandas_orders.csv')
customers = pd.read_csv('data/part01_pandas/pandas_customers.csv')
print(orders.shape)
print(customers.shape)

(200, 10)
(60, 4)


## 01 판다스 집계와 계산

## 02 기초 통계 함수

### 1 계산 대상 확인

통계를 계산하기 전에 어떤 컬럼이 수치형인지 먼저 확인한다. head( )로 값의 생김새를 보고 dtypes로 자료형을 함께 확인하면 계산 대상을 정확히 정할 수 있다.

In [2]:
cols = ['order_id', 'qty', 'unit_price', 'discount_rate', 'sales_amount']
print(orders[cols].head(6).to_string(index=False))
print(orders[['qty', 'unit_price', 'discount_rate', 'sales_amount']].dtypes)

order_id  qty  unit_price  discount_rate  sales_amount
  O00001   11       29100           0.05        304095
  O00002   11       91700           0.00       1008700
  O00003    2       60300           0.03        116982
  O00004    5       60500           0.00        302500
  O00005    5       10000           0.00         50000
  O00006    2       98700           0.00        197400
qty                int64
unit_price         int64
discount_rate    float64
sales_amount       int64
dtype: object


### 2 합계, 평균, 중앙값과 범위

가장 기본이 되는 통계 함수부터 하나씩 적용해본다. sum( ), mean( ), median( ), min( ), max( )는 모두 시리즈에 바로 붙여 사용할 수 있다.

In [3]:
print(orders['sales_amount'].sum())
print(orders['sales_amount'].mean())
print(orders['sales_amount'].median())
print(orders['sales_amount'].min())
print(orders['sales_amount'].max())

75020932
375104.66
277067.5
12200
1313400


### 3 여러 통계량 한 번에 확인

통계 함수를 하나씩 호출하면 코드가 길어진다. agg( )에 함수 이름을 리스트로 전달하면 여러 컬럼의 여러 통계량을 한 표로 확인할 수 있다.

In [4]:
summary = orders[['qty', 'unit_price', 'sales_amount']].agg(
    ['min', 'max', 'mean', 'median']
).round(2)
print(summary.to_string())

          qty  unit_price  sales_amount
min      1.00      4900.0      12200.00
max     11.00    119400.0    1313400.00
mean     6.06     62120.5     375104.66
median   6.00     63150.0     277067.50


#### 2. describe( )로 전체 분포 확인

필요한 통계량을 일일이 지정하지 않고 분포 전체를 훑어보고 싶다면 describe( )를 사용한다.

In [5]:
desc = orders[['qty', 'unit_price', 'sales_amount']].describe().round(2)
print(desc.to_string())

          qty  unit_price  sales_amount
count  200.00      200.00        200.00
mean     6.06    62120.50     375104.66
std      3.18    33263.53     317950.73
min      1.00     4900.00      12200.00
25%      3.00    34525.00     113252.00
50%      6.00    63150.00     277067.50
75%      9.00    92000.00     571125.00
max     11.00   119400.00    1313400.00


### 4 조건을 적용한 계산

통계량은 계산으로 끝나지 않고 조건과 결합되는 경우가 많다. 평균 이상인 주문이 몇 건인지 세려면 먼저 평균을 구한 뒤 이를 조건식에 사용한다.

In [6]:
mean_sales = orders['sales_amount'].mean()
cond = orders['sales_amount'] >= mean_sales
print(mean_sales)
print(cond.sum())

375104.66
75


## 03 value_counts

### 1 기본 빈도 계산

회원등급과 가입채널 컬럼에 value_counts( )를 적용해 각 값이 몇 번 등장하는지 센다.

In [7]:
print(customers['member_grade'].value_counts().to_string())
print(customers['join_channel'].value_counts().to_string())

member_grade
basic     26
silver    21
vip        7
gold       6
join_channel
app        26
web        21
offline    13


### 2 인덱스 순서와 비율

기본 정렬은 개수 기준이므로 10대, 20대 같은 순서가 뒤섞일 수 있다. 이럴 때는 sort_index( )로 순서를 정리하고, 비율이 필요하면 normalize=True를 사용한다.

In [8]:
age_count = customers['age_group'].value_counts().sort_index()
ratio = customers['join_channel'].value_counts(normalize=True).round(3)
print(age_count.to_string())
print(ratio.to_string())

age_group
10대        4
20대       19
30대       21
40대       12
50대 이상     4
join_channel
app        0.433
web        0.350
offline    0.217


### 3 조건을 적용한 빈도

전체가 아니라 특정 조건에 해당하는 데이터 안에서의 분포를 보고 싶을 때도 있다. app으로 가입한 고객만 골라낸 뒤 회원등급 빈도를 계산해본다.

In [9]:
app_customers = customers[customers['join_channel'] == 'app']
print(app_customers['member_grade'].value_counts().to_string())

member_grade
silver    10
basic      9
vip        4
gold       3


### 4 두 컬럼의 조합 빈도

한 컬럼씩 따로 보는 대신 두 컬럼의 조합이 몇 번 나타나는지 셀 수도 있다.

In [10]:
combo = customers[['member_grade', 'join_channel']].value_counts()
print(combo.head(8).to_string())

member_grade  join_channel
silver        app             10
basic         offline          9
              app              9
              web              8
silver        web              8
vip           app              4
gold          app              3
silver        offline          3


## 04 groupby

### 1 그룹별 합계

지역별 매출 합계부터 계산해본다. 묶을 컬럼은 region, 계산할 컬럼은 sales_amount, 계산 함수는 sum( )이다.

In [11]:
region_sales = orders.groupby('region')['sales_amount'].sum()
print(region_sales.sort_values(ascending=False).to_string())

region
강원권    15230717
제주권    13563540
호남권    13529997
영남권    13118465
수도권    12078330
충청권     7499883


### 2 그룹별 행 수와 평균

합계 외에도 그룹별 주문 건수나 평균이 필요한 경우가 많다. 행 수는 size( ), 평균은 mean( )으로 계산한다.

In [12]:
channel_count = orders.groupby('channel').size()
category_qty = orders.groupby('category')['qty'].mean().round(2)
print(channel_count.sort_values(ascending=False).to_string())
print(category_qty.to_string())

channel
app      80
web      53
store    50
phone    17
category
beauty     5.17
book       5.97
digital    5.94
kitchen    6.90
office     6.15
sports     6.34


### 3 여러 기준으로 묶기

기준이 하나일 필요는 없다. groupby( )에 컬럼을 리스트로 전달하면 두 기준의 조합별로 묶을 수 있다.

In [13]:
combo_sales = orders.groupby(['region', 'category'])['sales_amount'].sum()
print(combo_sales.sort_values(ascending=False).head(8).to_string())

region  category
호남권     office      4584400
영남권     kitchen     4140623
수도권     book        3584369
호남권     sports      3423940
수도권     sports      3371279
강원권     digital     3290272
        office      3268410
        kitchen     2914100


### 4 결과를 데이터프레임으로 만들기

groupby( ) 결과는 기준 컬럼이 인덱스로 들어간 시리즈 형태라 후속 작업이 번거로울 수 있다. as_index=False를 사용하면 처음부터 데이터프레임 형태로 결과를 받을 수 있다.

In [14]:
region_table = orders.groupby('region', as_index=False)['sales_amount'].sum()
region_table = region_table.sort_values('sales_amount', ascending=False)
print(region_table.to_string(index=False))

region  sales_amount
   강원권      15230717
   제주권      13563540
   호남권      13529997
   영남권      13118465
   수도권      12078330
   충청권       7499883


## 05 agg

### 1 한 컬럼에 여러 함수 적용

주문금액과 주문 수량 각각에 필요한 함수 목록을 전달해 여러 통계량을 한 번에 계산한다.

In [15]:
sales_summary = orders['sales_amount'].agg(['sum', 'mean', 'max'])
qty_summary = orders['qty'].agg(['min', 'max', 'mean', 'sum'])
print(sales_summary.round(2).to_string())
print(qty_summary.round(2).to_string())

sum     75020932.00
mean      375104.66
max      1313400.00
min        1.00
max       11.00
mean       6.06
sum     1211.00


### 2 groupby와 agg 결합

groupby( ) 뒤에 agg( )를 이어 붙이면 그룹별로 여러 통계량을 한 표에 담을 수 있다. 채널별 주문 건수, 합계, 평균, 최댓값을 함께 계산해본다.

In [16]:
channel_summary = orders.groupby('channel')['sales_amount'].agg(
    ['count', 'sum', 'mean', 'max']
).round(1)
print(channel_summary.to_string())

         count       sum      mean      max
channel                                    
app         80  32723394  409042.4  1313400
phone       17   5312283  312487.2   951900
store       50  16906597  338131.9  1062100
web         53  20078658  378842.6  1248390


### 3 여러 컬럼에 서로 다른 함수 적용

컬럼마다 적용할 함수가 다르다면 agg( )에 딕셔너리를 전달한다.

In [17]:
category_summary = orders.groupby('category').agg({
    'qty': ['sum', 'mean'],
    'sales_amount': ['sum', 'mean']
}).round(2)
print(category_summary.to_string())

          qty       sales_amount           
          sum  mean          sum       mean
category                                   
beauty    186  5.17      9397217  261033.81
book      197  5.97     13201466  400044.42
digital   184  5.94     10560525  340662.10
kitchen   214  6.90     13917022  448936.19
office    246  6.15     15394031  384850.78
sports    184  6.34     12550671  432781.76


### 4 이름 붙인 집계

집계 결과의 컬럼명이 sum, mean처럼 함수 이름 그대로면 의미를 알아보기 어렵다. 이름 붙인 집계를 사용하면 결과 컬럼명을 직접 정할 수 있다.

In [18]:
region_summary = orders.groupby('region').agg(
    order_count=('order_id', 'count'),
    total_qty=('qty', 'sum'),
    total_sales=('sales_amount', 'sum'),
    avg_sales=('sales_amount', 'mean')
).round(1)
print(region_summary.sort_values('total_sales', ascending=False).to_string())

        order_count  total_qty  total_sales  avg_sales
region                                                
강원권              38        246     15230717   400808.3
제주권              30        191     13563540   452118.0
호남권              35        209     13529997   386571.3
영남권              38        218     13118465   345222.8
수도권              35        220     12078330   345095.1
충청권              24        127      7499883   312495.1


## 06 pivot_table

### 1 한 개 기준으로 피벗테이블 만들기

가장 단순한 형태부터 만들어본다. 기준 하나에 계산할 컬럼 하나를 지정하면 groupby( )와 같은 결과를 표 형태로 얻을 수 있다.

In [19]:
region_sales = pd.pivot_table(
    orders,
    values='sales_amount',
    index='region',
    aggfunc='sum'
)
print(region_sales.to_string())

        sales_amount
region              
강원권         15230717
수도권         12078330
영남권         13118465
제주권         13563540
충청권          7499883
호남권         13529997


### 2 행과 열에 기준 배치

In [20]:
region_channel = pd.pivot_table(
    orders,
    values='sales_amount',
    index='region',
    columns='channel',
    aggfunc='sum',
    fill_value=0
)
print(region_channel.to_string())

channel      app    phone    store      web
region                                     
강원권      5900832  2083900  2661390  4584595
수도권      5376029   420700  3198115  3083486
영남권      3411744   807080  2970500  5929141
제주권      7998749  1137392  3170673  1256726
충청권      2479303   279216  2130754  2610610
호남권      7556737   583995  2775165  2614100


### 3 평균과 여러 값 요약

합계 대신 평균이 필요하면 aggfunc만 바꾸면 된다. 상품 분류와 채널 조합별 평균 주문 수량을 계산해본다.

In [21]:
qty_pivot = pd.pivot_table(
    orders,
    values='qty',
    index='category',
    columns='channel',
    aggfunc='mean'
).round(2)
print(qty_pivot.to_string())

channel    app  phone  store   web
category                          
beauty    5.25   6.00   5.50  4.12
book      5.31    NaN   6.25  6.89
digital   5.93  11.00   4.71  6.38
kitchen   7.55   6.75   6.00  6.89
office    7.53   6.80   4.80  5.10
sports    5.82   9.67   5.83  6.22


#### 2. 여러 값에 서로 다른 계산 적용

agg( )에서 딕셔너리를 사용했던 것처럼, 피벗테이블에서도 aggfunc에 딕셔너리를 전달하면 컬럼마다 다른 계산을 적용할 수 있다.

In [22]:
category_summary = pd.pivot_table(
    orders,
    values=['qty', 'sales_amount'],
    index='category',
    aggfunc={'qty': 'sum', 'sales_amount': 'mean'}
).round(1)
print(category_summary.to_string())

          qty  sales_amount
category                   
beauty    186      261033.8
book      197      400044.4
digital   184      340662.1
kitchen   214      448936.2
office    246      384850.8
sports    184      432781.8


### 4 전체 합계 표시

조합별 값과 함께 행과 열의 전체 합계를 같이 보고 싶다면 margins=True를 추가한다.

In [23]:
sales_pivot = pd.pivot_table(
    orders,
    values='sales_amount',
    index='region',
    columns='channel',
    aggfunc='sum',
    fill_value=0,
    margins=True
)
print(sales_pivot.to_string())

channel       app    phone     store       web       All
region                                                  
강원권       5900832  2083900   2661390   4584595  15230717
수도권       5376029   420700   3198115   3083486  12078330
영남권       3411744   807080   2970500   5929141  13118465
제주권       7998749  1137392   3170673   1256726  13563540
충청권       2479303   279216   2130754   2610610   7499883
호남권       7556737   583995   2775165   2614100  13529997
All      32723394  5312283  16906597  20078658  75020932


## 07 apply와 map

### 1 map으로 값 매핑

map( )은 딕셔너리로 만든 대응표에 따라 값을 다른 값으로 바꾸는 함수이다. 영문 채널명을 한글로 바꾸는 대응표를 만들어 적용해본다.

In [24]:
# 원본 orders를 유지하기 위해 복사본을 사용한다.
orders_work = orders.copy()
channel_name = {
    'app': '앱',
    'web': '웹',
    'store': '매장',
    'phone': '전화'
}
orders_work['channel_name'] = orders_work['channel'].map(channel_name)
print(orders_work[['channel', 'channel_name']].head(8).to_string(index=False))

channel channel_name
  phone           전화
    web            웹
    app            앱
    web            웹
    web            웹
    app            앱
    web            웹
    web            웹


#### 1. 상품 분류 이름 변환

같은 방법을 상품 분류 컬럼에도 적용해본다. 대응표만 새로 만들면 코드 구조는 동일하다.

In [25]:
category_name = {
    'digital': '디지털', 'kitchen': '주방', 'office': '사무',
    'beauty': '뷰티', 'book': '도서', 'sports': '스포츠'
}
orders_work['category_ko'] = orders_work['category'].map(category_name)
print(orders_work[['category', 'category_ko']].head(6).to_string(index=False))

category category_ko
 digital         디지털
 kitchen          주방
 digital         디지털
 kitchen          주방
  office          사무
  sports         스포츠


### 2 apply로 조건 적용

map( )이 정해진 대응표로 값을 바꾼다면, apply( )는 함수를 적용해 값을 만들어낸다. 할인율이 0보다 크면 '할인', 아니면 '정가'로 구분하는 조건을 lambda 함수로 적용해본다.

In [26]:
orders_work['discount_flag'] = orders_work['discount_rate'].apply(
    lambda x: '할인' if x > 0 else '정가'
)
print(orders_work['discount_flag'].value_counts().to_string())

discount_flag
할인    114
정가     86


### 3 행 단위 apply

지금까지는 컬럼 하나의 값만 사용했지만, 계산에 여러 컬럼이 필요할 때도 있다. 수량과 단가를 곱해 할인 전 금액을 구하려면 apply( )에 axis=1을 지정해 행 단위로 함수를 적용한다.

In [27]:
def calc_raw(row):
    return row['qty'] * row['unit_price']
orders_work['raw_amount'] = orders_work.apply(calc_raw, axis=1)
print(orders_work[['order_id', 'qty', 'unit_price', 'raw_amount', 'sales_amount']].head(
).to_string(index=False))

order_id  qty  unit_price  raw_amount  sales_amount
  O00001   11       29100      320100        304095
  O00002   11       91700     1008700       1008700
  O00003    2       60300      120600        116982
  O00004    5       60500      302500        302500
  O00005    5       10000       50000         50000


### 4 함수를 만들어 여러 조건 처리

조건이 두 갈래를 넘어가면 lambda 한 줄로는 읽기 어려워진다. 주문금액을 고액, 중간, 소액 세 단계로 나누는 함수를 따로 정의해 apply( )에 전달해본다.

In [28]:
def price_level(x):
    if x >= 700000:
        return '고액'
    elif x >= 200000:
        return '중간'
    else:
        return '소액'
orders_work['sales_level'] = orders_work['sales_amount'].apply(price_level)
print(orders_work['sales_level'].value_counts().to_string())

sales_level
중간    88
소액    78
고액    34


## 08 데이터프레임 병합

### 1 병합 기준 확인

병합하기 전에 두 데이터의 크기와 컬럼 구성을 확인한다. 어떤 컬럼을 기준으로 연결할지 눈으로 먼저 찾아야 한다.

In [29]:
print(orders.shape)
print(customers.shape)
print('orders 컬럼: ', list(orders.columns))
print('customers 컬럼: ', list(customers.columns))

(200, 10)
(60, 4)
orders 컬럼:  ['order_id', 'customer_id', 'region', 'channel', 'category', 'qty', 'unit_price', 'discount_rate', 'order_date', 'sales_amount']
customers 컬럼:  ['customer_id', 'age_group', 'member_grade', 'join_channel']


### 2 left 병합

customer_id를 기준으로 orders에 고객 정보를 붙여 본다. 주문 데이터를 기준으로 삼을 것이므로 how='left'를 지정한다.

In [30]:
merged = pd.merge(
    orders,
    customers,
    on='customer_id',
    how='left'
)
print(merged.shape)
print(merged[['order_id', 'customer_id', 'region', 'member_grade', 'age_group',
'sales_amount']].head(8).to_string(index=False))

(200, 13)


order_id customer_id region member_grade age_group  sales_amount
  O00001       C0035    호남권        basic       40대        304095
  O00002       C0001    호남권       silver       20대       1008700
  O00003       C0010    충청권        basic       30대        116982
  O00004       C0024    강원권          vip       40대        302500
  O00005       C0045    호남권        basic       10대         50000
  O00006       C0035    수도권        basic       40대        197400
  O00007       C0058    제주권        basic       20대         48060
  O00008       C0033    수도권          vip       30대         53900


### 3 필요한 컬럼만 병합

고객 정보 전체가 필요하지 않을 때도 있다. 오른쪽 데이터에서 기준 컬럼과 필요한 컬럼만 골라낸 뒤 병합하면 결과가 가벼워진다.

In [31]:
customer_small = customers[['customer_id', 'member_grade']]
merged_small = orders.merge(customer_small, on='customer_id', how='left')
print(merged_small.shape)
print(merged_small[['order_id', 'customer_id', 'member_grade',
'sales_amount']].head(8).to_string(index=False))

(200, 11)
order_id customer_id member_grade  sales_amount
  O00001       C0035        basic        304095
  O00002       C0001       silver       1008700
  O00003       C0010        basic        116982
  O00004       C0024          vip        302500
  O00005       C0045        basic         50000
  O00006       C0035        basic        197400
  O00007       C0058        basic         48060
  O00008       C0033          vip         53900


### 4 병합 결과 점검

병합이 끝나면 결과를 그대로 믿지 말고 점검한다. 행 수가 의도대로인지, 새로 붙인 컬럼에 결측치가 생기지 않았는지 확인한다.

In [32]:
print(merged.shape)
print(merged['member_grade'].isna().sum())
print(merged['member_grade'].value_counts().to_string())

(200, 13)
0
member_grade
silver    78
basic     77
vip       29
gold      16


### 5 left와 inner 비교

how 옵션에 따라 결과가 어떻게 달라지는지 left와 inner를 나란히 비교해본다.

In [33]:
left_merge = orders.merge(customers, on='customer_id', how='left')
inner_merge = orders.merge(customers, on='customer_id', how='inner')
print('left merge : ', left_merge.shape)
print('inner merge: ', inner_merge.shape)

left merge :  (200, 13)
inner merge:  (200, 13)


## 09 상관계수

### 1 숫자형 컬럼 선택

상관계수를 계산하기 전에 대상이 될 수치형 컬럼만 먼저 골라낸다.

In [34]:
num = orders[['qty', 'unit_price', 'discount_rate', 'sales_amount']]
print(num.head(6).to_string(index=False))

 qty  unit_price  discount_rate  sales_amount
  11       29100           0.05        304095
  11       91700           0.00       1008700
   2       60300           0.03        116982
   5       60500           0.00        302500
   5       10000           0.00         50000
   2       98700           0.00        197400

### 2 corr( )로 상관계수 표 만들기

선택한 데이터프레임에 corr( )를 적용하면 모든 컬럼 쌍의 상관계수를 표로 얻을 수 있다.

In [35]:
corr = num.corr()
print(corr.round(3).to_string())

                 qty  unit_price  discount_rate  sales_amount
qty            1.000       0.135          0.012         0.696
unit_price     0.135       1.000          0.043         0.730
discount_rate  0.012       0.043          1.000        -0.039
sales_amount   0.696       0.730         -0.039         1.000

### 3 두 컬럼만 계산

전체 표가 필요 없고 특정 두 컬럼의 관계만 궁금하다면 시리즈끼리 직접 corr( )를 사용한다.

In [36]:
print(orders['qty'].corr(orders['sales_amount']))
print(orders['unit_price'].corr(orders['sales_amount']))
print(orders['discount_rate'].corr(orders['sales_amount']))

0.6956149628784897
0.7300609337943281
-0.039132531197717486


### 4 기준 컬럼 중심으로 정렬

고 절댓값 기준으로 정렬해본다.

In [37]:
sales_corr = corr['sales_amount'].drop('sales_amount')
print(sales_corr.round(3).to_string())
print(sales_corr.abs().sort_values(ascending=False).round(3).to_string())

qty              0.696
unit_price       0.730
discount_rate   -0.039


unit_price       0.730
qty              0.696
discount_rate    0.039


## 10 분위수 계산

### 1 한 개의 분위수 계산

quantile( )에 0과 1 사이의 값을 전달하면 해당 위치의 분위수를 얻는다. 1사분위수부터 계산해본다.

In [38]:
q1 = orders['sales_amount'].quantile(0.25)
print(q1)

113252.0


### 2 여러 분위수를 한 번에 계산

분위수 위치를 리스트로 전달하면 여러 분위수를 한 번에 계산할 수 있다.

In [39]:
result = orders['sales_amount'].quantile([0.25, 0.5, 0.75])
print(result.to_string())

0.25    113252.0
0.50    277067.5
0.75    571125.0


### 3 여러 컬럼의 분위수

컬럼도 여러 개를 동시에 지정할 수 있다. 여러 컬럼을 선택한 데이터프레임에 quantile( )을 적용해본다.

In [40]:
cols = ['sales_amount', 'unit_price', 'qty']
result = orders[cols].quantile([0.25, 0.5, 0.75])
print(result.to_string())

      sales_amount  unit_price  qty
0.25      113252.0     34525.0  3.0
0.50      277067.5     63150.0  6.0
0.75      571125.0     92000.0  9.0


### 4 분위수를 기준으로 필터링

분위수는 계산으로 끝나지 않고 필터링 기준으로 이어지는 경우가 많다. 3사분위수를 구한 뒤 그 이상인 주문만 골라내 본다.

In [41]:
q3 = orders['sales_amount'].quantile(0.75)
high_orders = orders[orders['sales_amount'] >= q3]
print(q3)
print(len(high_orders))
print(high_orders[['order_id', 'category', 'sales_amount']].head().to_string(index=False))

571125.0
50
order_id category  sales_amount
  O00002  kitchen       1008700
  O00010     book       1249600
  O00012  digital        737100
  O00013     book       1248390
  O00026  kitchen        951900


## 11 반올림, 버림, 올림

### 1 금액 단위 변환

소수점 처리를 연습할 값을 먼저 만든다. 원 단위 주문금액을 만 원 단위로 나누면 소수점이 있는 값이 된다.

In [42]:
orders['amount_만원'] = orders['sales_amount'] / 10000
print(orders[['order_id', 'sales_amount', 'amount_만원']].head().to_string(index=False))

order_id  sales_amount  amount_만원
  O00001        304095    30.4095
  O00002       1008700   100.8700
  O00003        116982    11.6982
  O00004        302500    30.2500
  O00005         50000     5.0000


### 2 반올림

round( )에 자릿수를 전달하면 그 자리까지 반올림한다. 소수 첫째 자리로 반올림해본다.

In [43]:
orders['반올림_1자리'] = orders['amount_만원'].round(1)
print(orders[['order_id', 'amount_만원', '반올림_1자리']].head().to_string(index=False))

order_id  amount_만원  반올림_1자리
  O00001    30.4095     30.4
  O00002   100.8700    100.9
  O00003    11.6982     11.7
  O00004    30.2500     30.2
  O00005     5.0000      5.0


### 3 정수 자리 반올림

정수 자리로 반올림하려면 round(0)을 사용한다.

In [44]:
orders['반올림_0자리'] = orders['amount_만원'].round(0)
print(orders[['order_id', 'amount_만원', '반올림_0자리']].head().to_string(index=False))

order_id  amount_만원  반올림_0자리
  O00001    30.4095     30.0
  O00002   100.8700    101.0
  O00003    11.6982     12.0
  O00004    30.2500     30.0
  O00005     5.0000      5.0


### 4 버림과 올림

버림과 올림은 판다스가 아니라 넘파이 함수를 사용한다. 결과를 정수형으로 만들기 위해 astype(int)를 함께 적용한다.

In [45]:
orders['버림'] = np.floor(orders['amount_만원']).astype(int)
orders['올림'] = np.ceil(orders['amount_만원']).astype(int)
print(orders[['order_id', 'amount_만원', '버림', '올림']].head().to_string(index=False))

order_id  amount_만원  버림  올림
  O00001    30.4095  30  31
  O00002   100.8700 100 101
  O00003    11.6982  11  12
  O00004    30.2500  30  31
  O00005     5.0000   5   5


### 5 계산 결과 하나 처리하기

시험에서는 컬럼 전체가 아니라 평균 같은 값 하나를 처리해 제출하는 경우가 많다. 같은 평균값에 반올림, 버림, 올림을 각각 적용해 결과를 비교해본다.

In [46]:
avg = orders['sales_amount'].mean()
print(avg)
print(round(avg, 2))
print(round(avg))
print(int(np.floor(avg)))
print(int(np.ceil(avg)))

375104.66
375104.66
375105
375104
375105
